# 042 数据清洗：微博数据

In [1]:
TOPIC_WEIBO_PATH = r'..\data\fe\topic_weibo.parquet'
USER_WEIBO_PATH = r'..\data\fe\user_weibo.parquet'

TOPIC_COMMENT_PATH = r"..\data\cleaned\topic_comment.parquet"

In [2]:
import pandas as pd

df_topic_weibo = pd.read_parquet(TOPIC_WEIBO_PATH)
df_user_weibo = pd.read_parquet(USER_WEIBO_PATH)

df_topic_comment = pd.read_parquet(TOPIC_COMMENT_PATH)

## `df_user_weibo` 清洗

In [3]:
# ========== 1.1 去重 ==========
# user_weibo: 68,809 组同用户重复 weibo_id
# 策略：对于同一 weibo_id，保留第一条（同用户重复取其一）；
#        对于多用户同 weibo_id（82 组，转发关系），按 user_id 区分后保留
n_before = len(df_user_weibo)
df_user_weibo = df_user_weibo.drop_duplicates(subset=["weibo_id", "user_id"], keep="first")
print(f"\nuser_weibo 去重前：{n_before:,}  去重后: {len(df_user_weibo):>10,}")
print(f"  保留率: {len(df_user_weibo) / n_before * 100:.1f}%")


user_weibo 去重前：160,926  去重后:    147,251
  保留率: 91.5%


In [4]:
# 记录表原始行数，用于全程追踪保留率
_ORIGINAL_COUNTS = {
    "topic_weibo": len(df_topic_weibo),
    "user_weibo": len(df_user_weibo),
}

def retention_rate(table: str, current_df) -> str:
    """计算相对于原始数据的保留率。"""
    orig = _ORIGINAL_COUNTS[table]
    cur = len(current_df)
    return f"{cur:,} / {orig:,} ({cur / orig * 100:.1f}% 保留)"

print("\n📦 原始数据量:")
print(f"  topic_weibo:   {len(df_topic_weibo):>10,}")
print(f"  user_weibo:    {len(df_user_weibo):>10,}")


📦 原始数据量:
  topic_weibo:          222
  user_weibo:       147,251


In [5]:
import re

def clean_text(text: str) -> str:
    """清洗微博评论文本，保留情绪信号。

    清洗步骤（有序）：
    1. 移除 HTML 标签
    2. 移除 URL
    3. 规范化空白字符
    """
    if not isinstance(text, str) or len(text) == 0:
        return text

    # 1. 移除 HTML 标签
    text = re.sub(r'<[^>]+>', '', text)

    # 2. 移除 URL
    text = re.sub(r'https?://\S+', '', text)

    # 3. 规范化空白（多个空白合并为一个，去首尾空白）
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# 应用清洗
df_topic_weibo["content"] = df_topic_weibo["content"].apply(clean_text)
df_user_weibo["content"] = df_user_weibo["content"].apply(clean_text)

In [6]:
ad_keywords = [
    "分享有礼", "随机抽奖",
    "限时特卖",
    "领取优惠券", "购买请戳",
    "粉丝福利", "转发+评论"
]

ad_keyword = "粉丝福利"

df_user_weibo[df_user_weibo["content"].str.contains('|'.join(ad_keywords))]

,weibo_id,user_id,screen_name,content,text_length,text_quality,text_quality_label,create_time,year,month,...,hour,weekday,like_count,comment_count,repost_count,engagement,is_repost,reposted_weibo_id,topics,at_users
1076,5271231995314702,1056001684,龙哥in上海,豆豆粉丝福利,6,3,可分析,2026-02-28 09:45:05,2026,2,...,9,Saturday,0,0,0,0,True,5270326048987687,,
1271,5277782126888064,1420157965,上游新闻,#中超川渝德比门票免费送#【上游新闻的粉丝福利又来了！重庆铜梁龙VS成都蓉城球票免费送】#川...,394,3,可分析,2026-03-18 11:32:58,2026,3,...,11,Wednesday,5334,222,558,6114,False,-1,"中超川渝德比门票免费送,川渝德比久别重逢,上游新闻球迷观战团",
12700,5203798780019183,1630828973,撒满阳光,不错，来//@我是兔撕机:【抽奖】直接转发评论，抽一个华为耳机！原视频下评论猜一下最终起售价...,67,3,可分析,2025-08-26 07:49:33,2025,8,...,7,Tuesday,0,0,0,0,True,5203594796074949,,我是兔撕机
13941,5270638117783823,1653603955,扬子晚报,【扬子晚报送福利，当铁粉送你《镖人》限定周边 】春节档电影《镖人》热度持续飙升，吴京饰演的刀...,265,3,可分析,2026-02-26 18:25:14,2026,2,...,18,Thursday,30,33,31,94,False,-1,"镖人 一代大蠕,镖人 删减的打戏",扬子晚报
17328,5264121118589750,1136063370,晴天披沥,送福利，转发评论本微博，2月26日大年初十由 @微博抽奖平台 选一人送出我精心准备的新年礼品...,112,3,可分析,2026-02-08 18:49:00,2026,2,...,18,Sunday,1205,2088,2627,5920,False,-1,,微博抽奖平台
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150729,5208674457880350,5607510118,小牛电动,🔥小牛潮玩安家计划｜全网召集令🔥 🎁转发评论，抽1人送【只浪不费杜邦纸袋】！ 只要会玩❗️不...,208,3,可分析,2025-09-08 18:43:46,2025,9,...,18,Monday,89,89,91,269,False,-1,"玩新不止有点小牛,这很小牛,你有点小牛",
154195,5210058015117320,7912057882,用户1614482616,//@我是兔撕机:【抽奖】直接转发评论开一个华为耳机，原视频下评论“原创就是好”或“支持原创...,67,3,可分析,2025-09-12 14:21:32,2025,9,...,14,Friday,0,0,0,0,True,5209756960555391,,我是兔撕机
156773,5287694386857068,7937980855,小鱿鱼丝QAQ,"#时代少年团直播#溜溜梅专场直播（4月14日进行中） 内容亮点：成员探秘""天然梅冻工厂""，分...",136,3,可分析,2026-04-14 20:00:45,2026,4,...,20,Tuesday,0,1,0,1,False,-1,时代少年团直播,
157561,5279661286494889,7314727257,太湖湾音乐节,#太湖湾音乐节##喜力星银太湖湾音乐节##有热爱就有星朋友#喜力®星银®·第十二届太湖湾音乐...,447,3,可分析,2026-03-23 16:00:05,2026,3,...,16,Monday,266685,15163,39811,321659,False,-1,"太湖湾音乐节,喜力星银太湖湾音乐节,有热爱就有星朋友,安溥,太湖湾音乐节,喜力星银太湖湾音乐节","南青乐队,吴克群,万能青年旅店乐队,张远Bird,逃跑计划EscapePlan,陶喆,陆虎I..."


In [7]:
system_patterns = [
    "此微博已被作者删除",
    "微博可见时间范围",
    "没有这条微博的查看权限",
    "账号因违反相关法律法规",
    "该微博因违反法律法规",
    "被权利方投诉侵权",
    "用户自行申请关闭", 
    "微博社区公约", 
    "暂时无法查看", 
    "账号行为异常"
]

# 平台活动关键词
activity_keywords = [
    "点开红包", "现金红包", "微博红包", "随机抽奖",
    "抽奖详情", "领取优惠券", "购买请戳", "限时特卖",
    "分享有礼", "试试你的手气", "试手气", "抽奖平台", 
    "转发评论", "转发+评论", "转发关注", "转发+关注", 
    "关注转发", "关注+转发", "转关", "转+关", "好礼", 
    "年度歌曲", "我在参与", "免费围观", "森林驿站", 
    "开放公测", "上闲鱼", "微博智搜", "微博抓马", 
    "春节AI合拍", "微博渔场", "解锁赛博年味", "年度报告", 
    "旅行青蛙中国", "微博之夜", "粉丝福利", "好运在此", 
    "运气好到爆", "嗨抢", "欧气爆棚", "抓马福", "马年接福", 
    "微博回忆", "集福袋"
]

# 签到打卡关键词
streak_keywords = [
    "连续签到", "粉打卡", "关注超话", "签到活动", "集卡", 
    "头像挂件", "SVIP", "微博会员", 
]
print("✅ 系统提示词 & 广告关键词 已定义")

✅ 系统提示词 & 广告关键词 已定义


In [8]:
df_user_weibo[df_user_weibo["text_quality"] == 3]["content"].value_counts()

content
转发微博                                                                                                                      23532
                                                                                                                           2321
抱歉，根据作者设置的微博可见时间范围，此微博已不可见。                                                                                                1393
抱歉，此微博已被作者删除。查看帮助： 网页链接                                                                                                     457
抱歉，由于作者设置，你暂时没有这条微博的查看权限哦。查看帮助： 网页链接                                                                                        404
                                                                                                                          ...  
老铿的人生哲学：允许≠放任默认，顺其自然的前提是争其必然                                                                                                  1
拜托不要再“打听”一双关系好不好了 牛顿要是还活着都得提出第四定律 “王昶梁伟铿天下第一好”公式是WCx LWK=LOVE                                   

In [9]:
df_user_weibo[df_user_weibo["content"].str.contains(r"集福袋", regex=True) & (df_user_weibo["text_quality"] == 3)]

,weibo_id,user_id,screen_name,content,text_length,text_quality,text_quality_label,create_time,year,month,...,hour,weekday,like_count,comment_count,repost_count,engagement,is_repost,reposted_weibo_id,topics,at_users
3336,5249327875233456,1146661944,Theonlywinegood,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-29 23:05:56,2025,12,...,23,Monday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
3338,5248604260536139,1146661944,Theonlywinegood,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-27 23:10:33,2025,12,...,23,Saturday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
3339,5247881607905600,1146661944,Theonlywinegood,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-25 23:18:58,2025,12,...,23,Thursday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
3351,5247514360156043,1146661944,Theonlywinegood,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-24 22:59:40,2025,12,...,22,Wednesday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
20660,5250067124460642,1771817560,锅里有你_,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2026-01-01 00:03:27,2026,1,...,0,Thursday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
68127,5249935094056202,5342875988,Remember丶菜,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-31 15:18:48,2025,12,...,15,Wednesday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
87996,5248094394123230,5928177857,最初的心动ya,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-26 13:24:31,2025,12,...,13,Friday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
100656,5248954888626791,6298771910,温德米尔夫人,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-28 22:23:49,2025,12,...,22,Sunday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
129575,5250021748117450,7510103953,姜丹丽,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-31 21:03:08,2025,12,...,21,Wednesday,7,0,0,7,False,-1,"2025为自己颁奖,你好2026",


In [10]:
# ========== 3. 文本质量分级机制 ==========
# 本阶段为 df_user_weibo 添加文本质量等级字段，用于区分微博文本分析价值
# 
# 级别定义：
# - Level 0 "系统提示" / "空内容"：无有效用户表达
#   - "系统提示"：平台生成的系统提示（删帖、权限等）
#   - "空内容"：清洗后为空的微博文本
# - Level 1 "广告、抽奖、营销等"：虽为用户文本，但主要用于商业推广、活动参与，模板化程度高
# - Level 2 "低信息量"：属于用户文本，但内容极少，信息密度极低（纯数字/符号/字母/Emoji）
# - Level 3 "普通内容"：具有基本语义内容的普通微博，可供后续情绪分析使用
# - Level 4 "高质量内容"：当前不需要识别（暂留作未来扩展）

# ========== 3.1 初始化质量等级字段 ==========
df_user_weibo["text_quality"] = 3  # 默认设为 Level 3（普通内容）
df_user_weibo["text_quality_label"] = "普通内容"

print(f"✅ user_weibo 初始化质量等级字段")
print(f"   Shape: {df_user_weibo.shape}")

# ========== 3.2 定义质量判定函数 ==========

# 规则集：完全匹配的正则表达式（按优先级排序）
QUALITY_RULES = [
    # Level 1: 占位符和功能互动
    (r'^([转轉][发發]?(至?微博( 查看图片)?)?|Repost|分享(图片|新鲜事|视频)|网页链接|[存码马](克)?|转一个|签到|收藏)$', 1, '占位/功能互动'),
    (r'^#[^#]+#(\s*#[^#]+#)*$', 1, '纯话题占位'),
    
    # Level 2: 参与互动和祝福
    (r'^(我?来[了啦]{0,2}|接{1,3}|[抽中]|了解一下|收到|是的|关注|期待)$', 2, '参与互动'),
    (r'^((新年|元宵节|生日|除夕)快乐[！!]?|开工大吉|[早晚]安|早上好)$', 2, '固定祝福语'),
    (r'^(好(的|好好)?|哈{2,}|哇(哦)?|嗯)$', 2, '纯感叹'),
    (r'[\U0001F300-\U0001F9FF\U00002600-\U000027BF'
     r'\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF\uFE00-\uFE0F\u200D]+', 2, '纯表情')

]

def classify_text_quality(text: str) -> tuple:
    """根据微博文本内容，判定其质量等级。
    
    Args:
        text (str): 微博文本内容
    
    Returns:
        tuple: (quality_level: int, quality_label: str)
    """
    
    # 先检查是否为空内容（清洗后为空字符串）
    if not isinstance(text, str) or len(text.strip()) == 0:
        return (0, "空内容")
    
    t = text.strip()
    
    # Level 0：系统提示词（平台生成，无有效用户表达）
    for pattern in system_patterns:
        if pattern in t:
            return (0, "系统提示")
    
    # Level 1：广告、抽奖、营销等（模板化内容，商业导向）
    # 平台活动模板
    for keyword in activity_keywords:
        if keyword in t:
            return (1, "平台活动模板")
    # 签到打卡模板
    for keyword in streak_keywords:
        if keyword in t:
            return (1, "签到打卡模板")
    
    # ========== 新增：完全匹配的规则（按优先级） ==========
    for pattern, level, label in QUALITY_RULES:
        if re.fullmatch(pattern, t):
            return (level, label)

    # Level 2：低信息量文本（用户文本，但内容极少）
    if _is_low_info_text(t):
        return (2, "低信息量")
    
    # Level 3：普通内容（默认，具有基本语义内容）
    return (3, "普通内容")


def _is_low_info_text(text: str) -> bool:
    """判断文本是否为低信息量（纯数字/纯符号/纯英文字母/纯Emoji）。
    
    Args:
        text (str): 已去空格的文本
    
    Returns:
        bool: True 表示低信息量，False 表示有信息量
    """
    # 纯数字
    if re.fullmatch(r'\d+', text):
        return True
    # 纯符号（不含字母、数字、汉字、Emoji）
    if re.fullmatch(r'[^\w\u4e00-\u9fff\U00010000-\U0010FFFF]+', text):
        return True
    # 纯英文字母
    if re.fullmatch(r'[a-zA-Z]+', text):
        return True
    return False


# ========== 3.4 应用质量分级 ==========
print("\n\n📊 应用文本质量分级...")
df_user_weibo[["text_quality", "text_quality_label"]] = df_user_weibo["content"].apply(
    lambda x: pd.Series(classify_text_quality(x))
)

# 统计各质量等级的数量
quality_counts = df_user_weibo["text_quality_label"].value_counts()
print("\n📈 文本质量等级分布：")
for quality_idx in range(5):
    quality_labels = {
        0: "空内容 / 系统提示",
        1: "广告、抽奖、营销等",
        2: "低信息量 / 参与互动 / 祝福语 / 表情",
        3: "普通内容",
        4: "高质量内容"
    }
    label = quality_labels[quality_idx]
    count = len(df_user_weibo[df_user_weibo["text_quality"] == quality_idx])
    if count > 0:
        pct = count / len(df_user_weibo) * 100
        # 对于 Level 0，显示细分信息
        if quality_idx == 0:
            empty_count = len(df_user_weibo[df_user_weibo["text_quality_label"] == "空内容"])
            system_count = len(df_user_weibo[df_user_weibo["text_quality_label"] == "系统提示"])
            print(f"  Level {quality_idx} ({label:20s}): {count:>10,} ({pct:5.2f}%)")
            print(f"       ├─ 空内容: {empty_count:>10,}")
            print(f"       └─ 系统提示: {system_count:>10,}")
        else:
            print(f"  Level {quality_idx} ({label:20s}): {count:>10,} ({pct:5.2f}%)")

# 显示新增规则的分布
print("\n📊 Level 1 标签分布（新增规则）:")
level1_labels = df_user_weibo[df_user_weibo["text_quality"] == 1]["text_quality_label"].value_counts()
for label, count in level1_labels.items():
    pct = count / len(df_user_weibo[df_user_weibo["text_quality"] == 1]) * 100
    print(f"  {label:20s}: {count:>10,} ({pct:5.2f}%)")

print("\n📊 Level 2 标签分布（新增规则）:")
level2_labels = df_user_weibo[df_user_weibo["text_quality"] == 2]["text_quality_label"].value_counts()
for label, count in level2_labels.items():
    pct = count / len(df_user_weibo[df_user_weibo["text_quality"] == 2]) * 100
    print(f"  {label:20s}: {count:>10,} ({pct:5.2f}%)")

print(f"\n✅ 文本质量分级完成: {retention_rate('user_weibo', df_user_weibo)}")


✅ user_weibo 初始化质量等级字段
   Shape: (147251, 21)


📊 应用文本质量分级...

📈 文本质量等级分布：
  Level 0 (空内容 / 系统提示          ):      4,966 ( 3.37%)
       ├─ 空内容:      2,321
       └─ 系统提示:      2,645
  Level 1 (广告、抽奖、营销等           ):     32,109 (21.81%)
  Level 2 (低信息量 / 参与互动 / 祝福语 / 表情):        908 ( 0.62%)
  Level 3 (普通内容                ):    109,268 (74.21%)

📊 Level 1 标签分布（新增规则）:
  占位/功能互动             :     24,329 (75.77%)
  平台活动模板              :      6,257 (19.49%)
  纯话题占位               :      1,270 ( 3.96%)
  签到打卡模板              :        253 ( 0.79%)

📊 Level 2 标签分布（新增规则）:
  低信息量                :        321 (35.35%)
  纯感叹                 :        212 (23.35%)
  纯表情                 :        171 (18.83%)
  参与互动                :        126 (13.88%)
  固定祝福语               :         78 ( 8.59%)

✅ 文本质量分级完成: 147,251 / 147,251 (100.0% 保留)


## `df_topic_weibo` 清洗

In [11]:
# ========== 1.1 去重 ==========
# user_weibo: 68,809 组同用户重复 weibo_id
# 策略：对于同一 weibo_id，保留第一条（同用户重复取其一）；
#        对于多用户同 weibo_id（82 组，转发关系），按 user_id 区分后保留
n_before = len(df_topic_weibo)
df_topic_weibo = df_topic_weibo.drop_duplicates(subset=["weibo_id", "user_id"], keep="first")
print(f"\ntopic_weibo 去重前：{n_before:,}  去重后: {len(df_topic_weibo):>10,}")
print(f"  保留率: {len(df_topic_weibo) / n_before * 100:.1f}%")


topic_weibo 去重前：222  去重后:        202
  保留率: 91.0%


In [12]:
df_topic_weibo = pd.read_parquet(TOPIC_WEIBO_PATH)


# ========== 4. 评论聚合统计并回填到 topic_weibo ==========
print("\n\n📊 开始评论聚合统计...")

# 4.1 按 weibo_id 聚合评论数据
comment_agg = df_topic_comment.groupby("weibo_id").agg(
    comment_crawled_count=("weibo_id", "size"),  # 爬取到的评论总数
    comment_hq_count=("text_quality", lambda x: (x >= 3).sum()),  # 高质量评论数
    comment_hq_user_count=("user_id", lambda x: x[df_topic_comment.loc[x.index, "text_quality"] >= 3].nunique())  # 高质量评论的去重用户数
).reset_index()

# 4.2 计算高质量评论比例
comment_agg["comment_hq_ratio"] = (comment_agg["comment_hq_count"] / comment_agg["comment_crawled_count"]).round(2)

print(f"\n✅ 评论聚合统计完成：{len(comment_agg):,} 个微博")
print(f"   字段：{list(comment_agg.columns)}")

# 4.3 将聚合结果回填到 df_topic_weibo
df_topic_weibo = df_topic_weibo.merge(
    comment_agg[["weibo_id", "comment_crawled_count", "comment_hq_count", "comment_hq_ratio", "comment_hq_user_count"]],
    on="weibo_id",
    how="left"
)

# 处理没有评论的微博（填充为 0 或相关值）
df_topic_weibo["comment_crawled_count"] = df_topic_weibo["comment_crawled_count"].fillna(0).astype(int)
df_topic_weibo["comment_hq_count"] = df_topic_weibo["comment_hq_count"].fillna(0).astype(int)
df_topic_weibo["comment_hq_ratio"] = df_topic_weibo["comment_hq_ratio"].fillna(0.0).round(2)
df_topic_weibo["comment_hq_user_count"] = df_topic_weibo["comment_hq_user_count"].fillna(0).astype(int)

print(f"\n✅ 评论统计已回填到 df_topic_weibo")

# ========== 4.4 计算话题价值等级 ==========
print("\n📊 开始计算话题价值等级...")

def calculate_topic_value(row):
    """根据评论统计信息，判定话题的价值等级。
    
    规则：
    - comment_crawled_count < 15: Level 0 "评论样本不足"
    - comment_crawled_count >= 15 且 comment_hq_count < 17: Level 1 "低价值"
    - comment_crawled_count >= 20 且 comment_hq_count >= 27 且 comment_hq_user_count >= 24: Level 3 "高价值"
    - 其余: Level 2 "一般价值"
    
    Args:
        row: DataFrame 的一行
    
    Returns:
        tuple: (topic_value: int, topic_value_label: str)
    """
    crawled = row["comment_crawled_count"]
    hq_count = row["comment_hq_count"]
    hq_user = row["comment_hq_user_count"]
    trending_type = row["trending_type"]
    
    if crawled < 15:
        return (0, "评论样本不足")
    elif crawled >= 15 and hq_count < 17:
        return (1, "低价值")
    elif trending_type is not None and crawled >= 20 and hq_count >= 27 and hq_user >= 24:
        return (3, "高价值")
    else:
        return (2, "一般价值")

# 应用话题价值计算
df_topic_weibo[["topic_value", "topic_value_label"]] = df_topic_weibo.apply(
    lambda row: pd.Series(calculate_topic_value(row)),
    axis=1
)

# 统计话题价值分布
print(f"\n📈 话题价值等级分布：")
value_counts = df_topic_weibo["topic_value_label"].value_counts().sort_index()
for label in ["评论样本不足", "低价值", "一般价值", "高价值"]:
    count = len(df_topic_weibo[df_topic_weibo["topic_value_label"] == label])
    if count > 0:
        pct = count / len(df_topic_weibo) * 100
        print(f"  {label:15s}: {count:>10,} ({pct:5.2f}%)")

print(f"\n✅ 话题价值等级计算完成")

# 4.5 调整列顺序
topic_weibo_cols = [
    # ID & 用户
    "weibo_id", "user_id", "screen_name", "gender",
    # 话题
    "topic",
    # 文本
    "content", "text_length",
    # 时间
    "create_time", "year", "month", "day", "hour", "weekday",
    # 互动
    "like_count", "comment_count", "repost_count", "engagement",
    # 爬取评论信息
    "comment_crawled_count", "comment_hq_count", "comment_hq_ratio", "comment_hq_user_count",
    # 话题价值评估
    "topic_value", "topic_value_label",
    # 热搜词条信息
    "trending_date", "trending_type", "trending_click"
]

df_topic_weibo = df_topic_weibo[topic_weibo_cols]

print(f"\n✅ 列顺序已调整")
print(f"   新的 df_topic_weibo shape: {df_topic_weibo.shape}")
print(f"   新的 df_topic_weibo columns: {list(df_topic_weibo.columns)}")



📊 开始评论聚合统计...

✅ 评论聚合统计完成：202 个微博
   字段：['weibo_id', 'comment_crawled_count', 'comment_hq_count', 'comment_hq_user_count', 'comment_hq_ratio']

✅ 评论统计已回填到 df_topic_weibo

📊 开始计算话题价值等级...

📈 话题价值等级分布：
  评论样本不足         :          3 ( 1.35%)
  一般价值           :          1 ( 0.45%)
  高价值            :        218 (98.20%)

✅ 话题价值等级计算完成

✅ 列顺序已调整
   新的 df_topic_weibo shape: (222, 26)
   新的 df_topic_weibo columns: ['weibo_id', 'user_id', 'screen_name', 'gender', 'topic', 'content', 'text_length', 'create_time', 'year', 'month', 'day', 'hour', 'weekday', 'like_count', 'comment_count', 'repost_count', 'engagement', 'comment_crawled_count', 'comment_hq_count', 'comment_hq_ratio', 'comment_hq_user_count', 'topic_value', 'topic_value_label', 'trending_date', 'trending_type', 'trending_click']


In [13]:
def sample_weibo_comments(level, n_weibo=50, n_comment=20) -> dict:
    """根据话题价值等级，从 df_topic_comment 中随机抽取微博评论样本。

    Args:
        level (int): 话题价值等级
        n_weibo (int): 抽样微博数量
        n_comment (int): 抽样评论数量

    Returns:
        dict
    """
    weibo_comment_dict = {}
    # 根据话题价值等级筛选微博
    level_weibo_count = len(df_topic_weibo[df_topic_weibo["topic_value"] == level])
    n_weibo = min(n_weibo, level_weibo_count)  # 确保抽样数量不超过可用微博数

    sampled_weibo_ids = df_topic_weibo[df_topic_weibo["topic_value"] == level].sample(n=n_weibo)["weibo_id"]
    for weibo_id in sampled_weibo_ids:
        comment_count = len(df_topic_comment[df_topic_comment["weibo_id"] == weibo_id])
        n_comment = min(n_comment, comment_count)  # 确保抽样数量不超过可用评论数
        sampled_comments = df_topic_comment[df_topic_comment["weibo_id"] == weibo_id][["screen_name", "content", "text_quality_label"]].sample(n=n_comment)
        weibo_comment_dict[weibo_id] = [(row["screen_name"], row["content"], row["text_quality_label"]) for _, row in sampled_comments.iterrows()]

    # 随机抽样
    return weibo_comment_dict

# sample_weibo_comments(3)

In [14]:
import os

# 创建输出目录
output_dir = r"..\data\cleaned"
os.makedirs(output_dir, exist_ok=True)

# 保存
datasets = {
    "topic_weibo": df_topic_weibo,
    "user_weibo": df_user_weibo,
}

for name, df in datasets.items():
    path = os.path.join(output_dir, f"{name}.parquet")
    df.to_parquet(path, index=False)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"✅ {name}.parquet 已保存 ({df.shape[0]:>10,} rows × {df.shape[1]:>2} cols, {size_mb:.1f} MB)")

print(f"\n📂 输出目录: {os.path.abspath(output_dir)}")

# 显示 user_weibo 的最终字段列表
print(f"\n📋 user_weibo 最终字段列表：")
print(f"  Columns: {list(df_user_weibo.columns)}")


✅ topic_weibo.parquet 已保存 (       222 rows × 26 cols, 0.1 MB)
✅ user_weibo.parquet 已保存 (   147,251 rows × 21 cols, 37.8 MB)

📂 输出目录: d:\GraduationProject\data\cleaned

📋 user_weibo 最终字段列表：
  Columns: ['weibo_id', 'user_id', 'screen_name', 'content', 'text_length', 'text_quality', 'text_quality_label', 'create_time', 'year', 'month', 'day', 'hour', 'weekday', 'like_count', 'comment_count', 'repost_count', 'engagement', 'is_repost', 'reposted_weibo_id', 'topics', 'at_users']
